In [1]:
"""
Gold Layer - Audit Logs Analytics & Anomaly Detection
Creates Delta tables in gitrepo.default schema
"""

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta

# ============================================================================
# CONFIGURATION
# ============================================================================

SILVER_PATH = "/Volumes/gitrepo/default/git_oci_aidp_silver/audit_logs/data"

# Gold tables - UPDATED to use gitrepo catalog
CATALOG = "gitrepo"
SCHEMA = "default"

GOLD_DAILY_SUMMARY = f"{CATALOG}.{SCHEMA}.gold_audit_daily_summary"
GOLD_USER_BASELINE = f"{CATALOG}.{SCHEMA}.gold_user_behavior_baseline"
GOLD_ANOMALY_INCIDENTS = f"{CATALOG}.{SCHEMA}.gold_audit_anomaly_incidents"
GOLD_FAILED_OPS = f"{CATALOG}.{SCHEMA}.gold_failed_operations_summary"

# Analysis parameters
LOOKBACK_DAYS = 60  # For baseline calculations
ANOMALY_THRESHOLD = 3.0  # Standard deviations for anomaly

print("=" * 70)
print("GOLD LAYER - ANALYTICS & ANOMALY DETECTION")
print("=" * 70)
print(f"Silver Path: {SILVER_PATH}")
print(f"Output Tables:")
print(f"  1. {GOLD_DAILY_SUMMARY}")
print(f"  2. {GOLD_USER_BASELINE}")
print(f"  3. {GOLD_ANOMALY_INCIDENTS}")
print(f"  4. {GOLD_FAILED_OPS}")
print("=" * 70)

# ============================================================================
# LOAD SILVER DATA
# ============================================================================

print("\n[STEP 1] Loading silver data...")

silver_df = spark.read.parquet(SILVER_PATH)
total_records = silver_df.count()

print(f"Loaded {total_records:,} records from silver layer")

# Filter to recent data for analysis
cutoff_date = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).date()
recent_df = silver_df.filter(F.col("ingest_date") >= F.lit(cutoff_date))

print(f"Analyzing last {LOOKBACK_DAYS} days: {recent_df.count():,} records")

# ============================================================================
# TABLE 1: DAILY SUMMARY AGGREGATIONS (Option 1)
# ============================================================================

print("\n[STEP 2] Creating daily summary aggregations...")

gold_daily = (
    silver_df
    .groupBy(
        "ingest_date",
        "compartment_name",
        "compartment_id",
        "event_name",
        "principal_name",
        "principal_id"
    )
    .agg(
        F.count("*").alias("event_count"),
        F.countDistinct("ip_address").alias("unique_ips"),
        F.countDistinct("resource_id").alias("unique_resources"),
        F.sum(F.when(F.col("response_status") == "200", 1).otherwise(0)).alias("success_count"),
        F.sum(F.when(F.col("response_status") != "200", 1).otherwise(0)).alias("failure_count"),
        F.collect_set("ip_address").alias("ip_addresses"),
        F.min("event_time").alias("first_event_time"),
        F.max("event_time").alias("last_event_time")
    )
    .withColumn("failure_rate", 
                F.round(F.col("failure_count") / F.col("event_count") * 100, 2))
    .withColumn("processing_timestamp", F.current_timestamp())
)

# Write as Delta table
print(f"Writing to {GOLD_DAILY_SUMMARY}...")

(gold_daily.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(GOLD_DAILY_SUMMARY))

daily_count = gold_daily.count()
print(f"✓ Created daily summary: {daily_count:,} records")

# ============================================================================
# TABLE 2: USER BEHAVIOR BASELINE (for anomaly detection)
# ============================================================================

print("\n[STEP 3] Creating user behavior baselines...")

# Calculate normal behavior patterns per user
gold_baseline = (
    recent_df
    .groupBy("principal_id", "principal_name", "event_name")
    .agg(
        F.count("*").alias("total_count"),
        F.countDistinct("ingest_date").alias("active_days"),
        F.avg(F.hour("event_time")).alias("avg_hour_of_day"),
        F.stddev(F.hour("event_time")).alias("stddev_hour"),
        F.countDistinct("compartment_id").alias("compartment_count"),
        F.countDistinct("ip_address").alias("ip_count"),
        F.collect_set("compartment_name").alias("typical_compartments"),
        F.collect_set("ip_address").alias("typical_ips")
    )
    .withColumn("avg_events_per_day", 
                F.round(F.col("total_count") / F.col("active_days"), 2))
    .withColumn("baseline_period_days", F.lit(LOOKBACK_DAYS))
    .withColumn("baseline_calculated_at", F.current_timestamp())
)

print(f"Writing to {GOLD_USER_BASELINE}...")

(gold_baseline.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(GOLD_USER_BASELINE))

baseline_count = gold_baseline.count()
print(f"✓ Created user baselines: {baseline_count:,} records")

# ============================================================================
# TABLE 3: ANOMALY DETECTION (Option 2)
# ============================================================================

print("\n[STEP 4] Detecting anomalies...")

# Calculate daily activity per principal+event
daily_activity = (
    recent_df
    .groupBy("ingest_date", "principal_id", "principal_name", "event_name", 
             "compartment_name", "ip_address")
    .agg(
        F.count("*").alias("daily_count"),
        F.max("response_status").alias("status")
    )
)

# Calculate statistics for each principal+event combination
window_spec = Window.partitionBy("principal_id", "event_name")

activity_with_stats = (
    daily_activity
    .withColumn("avg_count", F.avg("daily_count").over(window_spec))
    .withColumn("stddev_count", F.stddev("daily_count").over(window_spec))
    .withColumn("total_60d", F.sum("daily_count").over(window_spec))
)

# Detect anomalies: events that are X standard deviations from mean
anomalies = (
    activity_with_stats
    .withColumn("z_score", 
                (F.col("daily_count") - F.col("avg_count")) / 
                F.when(F.col("stddev_count") > 0, F.col("stddev_count")).otherwise(1))
    .filter(F.abs(F.col("z_score")) > ANOMALY_THRESHOLD)
    .withColumn("anomaly_type", 
                F.when(F.col("z_score") > 0, "unusually_high_activity")
                .otherwise("unusually_low_activity"))
    .withColumn("score", F.abs(F.col("z_score")))
    .select(
        F.current_timestamp().alias("run_time"),
        "ingest_date",
        "anomaly_type",
        "principal_id",
        "principal_name",
        "event_name",
        "compartment_name",
        "ip_address",
        "status",
        F.col("daily_count").alias("cnt_today"),
        F.round("avg_count", 2).alias("avg_60d"),
        F.round("stddev_count", 2).alias("stddev_60d"),
        "total_60d",
        F.round("score", 2).alias("score")
    )
    .orderBy(F.col("score").desc())
)

# Add additional anomaly types
# Rare events (first time seen)
rare_events = (
    recent_df
    .groupBy("principal_id", "principal_name", "event_name", "ingest_date")
    .agg(
        F.count("*").alias("cnt_today"),
        F.max("compartment_name").alias("compartment_name"),
        F.max("ip_address").alias("ip_address"),
        F.max("response_status").alias("status")
    )
    .join(
        gold_baseline.select("principal_id", "event_name", "total_count"),
        ["principal_id", "event_name"],
        "left"
    )
    .filter(F.col("total_count") < 5)  # Rare: less than 5 times in 60 days
    .withColumn("run_time", F.current_timestamp())
    .withColumn("anomaly_type", F.lit("rare_event_for_principal"))
    .withColumn("score", F.lit(5.0))  # High score for rare events
    .withColumn("avg_60d", F.lit(0.0))
    .withColumn("stddev_60d", F.lit(0.0))
    .withColumn("total_60d", F.coalesce(F.col("total_count"), F.lit(0)))
    .select(anomalies.columns)
)

# Combine all anomaly types
all_anomalies = anomalies.union(rare_events)

print(f"Writing to {GOLD_ANOMALY_INCIDENTS}...")

(all_anomalies.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(GOLD_ANOMALY_INCIDENTS))

anomaly_count = all_anomalies.count()
print(f"✓ Detected anomalies: {anomaly_count:,} incidents")

# ============================================================================
# TABLE 4: FAILED OPERATIONS SUMMARY
# ============================================================================

print("\n[STEP 5] Creating failed operations summary...")

gold_failed = (
    silver_df
    .filter(F.col("response_status") != "200")
    .groupBy(
        "ingest_date",
        "event_name",
        "principal_name",
        "principal_id",
        "compartment_name",
        "ip_address",
        "response_status"
    )
    .agg(
        F.count("*").alias("failure_count"),
        F.collect_list("response_message").alias("error_messages"),
        F.min("event_time").alias("first_failure"),
        F.max("event_time").alias("last_failure")
    )
    .withColumn("processing_timestamp", F.current_timestamp())
    .orderBy(F.col("failure_count").desc())
)

print(f"Writing to {GOLD_FAILED_OPS}...")

(gold_failed.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(GOLD_FAILED_OPS))

failed_count = gold_failed.count()
print(f"✓ Created failed operations summary: {failed_count:,} records")

# ============================================================================
# VERIFICATION & SUMMARY
# ============================================================================

print("\n" + "=" * 70)
print("GOLD LAYER TABLES CREATED")
print("=" * 70)

# Show sample from each table
print("\n1. Daily Summary (sample):")
spark.sql(f"""
    SELECT ingest_date, compartment_name, event_name, event_count, failure_rate
    FROM {GOLD_DAILY_SUMMARY}
    ORDER BY event_count DESC
    LIMIT 5
""").show(truncate=False)

print("\n2. Top Anomalies:")
spark.sql(f"""
    SELECT run_time, anomaly_type, principal_name, event_name, score, cnt_today, avg_60d
    FROM {GOLD_ANOMALY_INCIDENTS}
    ORDER BY score DESC
    LIMIT 10
""").show(truncate=False)

print("\n3. Top Failed Operations:")
spark.sql(f"""
    SELECT event_name, principal_name, ip_address, failure_count
    FROM {GOLD_FAILED_OPS}
    ORDER BY failure_count DESC
    LIMIT 5
""").show(truncate=False)

print("\n" + "=" * 70)
print("✓ GOLD LAYER PROCESSING COMPLETED")
print("=" * 70)
print("\nCreated Tables:")
print(f"  ✓ {GOLD_DAILY_SUMMARY} ({daily_count:,} records)")
print(f"  ✓ {GOLD_USER_BASELINE} ({baseline_count:,} records)")
print(f"  ✓ {GOLD_ANOMALY_INCIDENTS} ({anomaly_count:,} records)")
print(f"  ✓ {GOLD_FAILED_OPS} ({failed_count:,} records)")
print("\nQuery Examples:")
print(f"  SELECT * FROM {GOLD_ANOMALY_INCIDENTS} ORDER BY score DESC LIMIT 20")
print(f"  SELECT * FROM {GOLD_DAILY_SUMMARY} WHERE ingest_date = CURRENT_DATE()")
print("=" * 70)

GOLD LAYER - ANALYTICS & ANOMALY DETECTION
Silver Path: /Volumes/gitrepo/default/git_oci_aidp_silver/audit_logs/data
Output Tables:
  1. gitrepo.default.gold_audit_daily_summary
  2. gitrepo.default.gold_user_behavior_baseline
  3. gitrepo.default.gold_audit_anomaly_incidents
  4. gitrepo.default.gold_failed_operations_summary

[STEP 1] Loading silver data...


Loaded 17,383,006 records from silver layer


Analyzing last 60 days: 17,383,006 records

[STEP 2] Creating daily summary aggregations...
Writing to gitrepo.default.gold_audit_daily_summary...


✓ Created daily summary: 562,135 records

[STEP 3] Creating user behavior baselines...
Writing to gitrepo.default.gold_user_behavior_baseline...


✓ Created user baselines: 73,248 records

[STEP 4] Detecting anomalies...


...

✓ Detected anomalies: 58,806 incidents

[STEP 5] Creating failed operations summary...
Writing to gitrepo.default.gold_failed_operations_summary...


✓ Created failed operations summary: 20,718 records

GOLD LAYER TABLES CREATED

1. Daily Summary (sample):


+-----------+----------------+--------------------------------------------+-----------+------------+
|ingest_date|compartment_name|event_name                                  |event_count|failure_rate|
+-----------+----------------+--------------------------------------------+-----------+------------+
|2026-01-28 |sphinx          |io.k8s.coordination.v1.leases.update        |3614763    |0.0         |
|2026-01-28 |sphinx          |io.k8s.coordination.v1.leases.update        |2132032    |0.0         |
|2026-01-28 |kcflynn         |io.k8s.coordination.v1.leases.update        |1806778    |0.0         |
|2026-01-28 |kcflynn         |io.k8s.coordination.v1.leases.update        |1056340    |0.0         |
|2026-01-28 |sphinx          |io.k8s.authentication.v1.tokenreviews.create|725301     |100.0       |
+-----------+----------------+--------------------------------------------+-----------+------------+


2. Top Anomalies:


+--------------------------+-----------------------+-------------------+--------------+-----+---------+-------+
|run_time                  |anomaly_type           |principal_name     |event_name    |score|cnt_today|avg_60d|
+--------------------------+-----------------------+-------------------+--------------+-----+---------+-------+
|2026-01-28 03:08:58.518951|unusually_high_activity|Gustavo Saurez     |GetCompartment|26.89|176      |1.93   |
|2026-01-28 03:08:58.518951|unusually_high_activity|Christopher Johnson|GetCompartment|26.18|484      |5.38   |
|2026-01-28 03:08:58.518951|unusually_high_activity|Manasi Vaishampayan|ListPolicies  |20.72|32       |1.84   |
|2026-01-28 03:08:58.518951|unusually_high_activity|NULL               |ListDomains   |19.66|8        |1.07   |
|2026-01-28 03:08:58.518951|unusually_high_activity|NULL               |ListDomains   |19.66|8        |1.07   |
|2026-01-28 03:08:58.518951|unusually_high_activity|NULL               |ListPolicies  |19.31|8        |1

+--------------------------------------------+-----------------------------------------------------------------------------------------+--------------+-------------+
|event_name                                  |principal_name                                                                           |ip_address    |failure_count|
+--------------------------------------------+-----------------------------------------------------------------------------------------+--------------+-------------+
|io.k8s.authentication.v1.tokenreviews.create|oke                                                                                      |NULL          |725301       |
|io.k8s.authentication.v1.tokenreviews.create|oke                                                                                      |NULL          |360006       |
|DeletePar                                   |ocid1.aidataplatform.oc1.iad.amaaaaaac3adhhqav673ojt7xbm7iqw4sgf5o5i2g7odejhvn2cr6vuckktq|10.1.104.74   |43568        |
|Del

In [2]:
%sql
SELECT * FROM gitrepo.default.gold_audit_anomaly_incidents
ORDER BY score DESC
LIMIT 20

In [6]:
%sql
-- Daily activity overview
SELECT 
    ingest_date,
    compartment_name,
    event_name,
    SUM(event_count) as total_events,
    AVG(failure_rate) as avg_failure_rate,
    SUM(unique_ips) as total_unique_ips
FROM gitrepo.default.gold_audit_daily_summary
WHERE ingest_date >= DATE_SUB(CURRENT_DATE(), 7)
GROUP BY ingest_date, compartment_name, event_name
ORDER BY ingest_date DESC, total_events DESC
LIMIT 50

In [8]:
%sql
-- Most active users last 7 days
SELECT 
    principal_name,
    SUM(event_count) as total_events,
    AVG(failure_rate) as avg_failure_rate,
    COUNT(DISTINCT compartment_name) as compartments_accessed
FROM gitrepo.default.gold_audit_daily_summary
WHERE ingest_date >= DATE_SUB(CURRENT_DATE(), 7)
GROUP BY principal_name
ORDER BY total_events DESC
LIMIT 20

In [9]:
%sql
-- Compartment activity trends
SELECT 
    compartment_name,
    ingest_date,
    SUM(event_count) as events,
    SUM(failure_count) as failures,
    ROUND(SUM(failure_count) * 100.0 / SUM(event_count), 2) as failure_pct
FROM gitrepo.default.gold_audit_daily_summary
WHERE ingest_date >= DATE_SUB(CURRENT_DATE(), 30)
GROUP BY compartment_name, ingest_date
ORDER BY compartment_name, ingest_date DESC

In [10]:
%sql
-- High failure rate events
SELECT 
    event_name,
    compartment_name,
    principal_name,
    SUM(event_count) as attempts,
    SUM(failure_count) as failures,
    AVG(failure_rate) as avg_failure_rate
FROM gitrepo.default.gold_audit_daily_summary
WHERE ingest_date >= DATE_SUB(CURRENT_DATE(), 7)
GROUP BY event_name, compartment_name, principal_name
HAVING avg_failure_rate > 10
ORDER BY avg_failure_rate DESC, failures DESC
LIMIT 30

In [11]:
%sql
-- Most active users (baseline profile)
SELECT 
    principal_name,
    event_name,
    total_count,
    active_days,
    avg_events_per_day,
    compartment_count,
    ip_count,
    ROUND(avg_hour_of_day, 1) as typical_hour
FROM gitrepo.default.gold_user_behavior_baseline
ORDER BY total_count DESC
LIMIT 20

In [12]:
%sql
-- Users with unusual access patterns (many compartments/IPs)

SELECT 
    principal_name,
    COUNT(DISTINCT event_name) as unique_events,
    MAX(compartment_count) as max_compartments,
    MAX(ip_count) as max_ips,
    SUM(total_count) as total_events
FROM gitrepo.default.gold_user_behavior_baseline
GROUP BY principal_name
HAVING max_compartments > 5 OR max_ips > 5
ORDER BY total_events DESC
LIMIT 20

In [14]:
%sql
-- Event distribution by user
SELECT 
    event_name,
    COUNT(DISTINCT principal_id) as unique_users,
    SUM(total_count) as total_occurrences,
    AVG(avg_events_per_day) as avg_per_user_per_day
FROM gitrepo.default.gold_user_behavior_baseline
GROUP BY event_name
ORDER BY total_occurrences DESC
LIMIT 30

In [15]:
%sql
-- Off-hours activity (users active at unusual times)
SELECT 
    principal_name,
    event_name,
    ROUND(avg_hour_of_day, 1) as typical_hour,
    total_count,
    active_days
FROM gitrepo.default.gold_user_behavior_baseline
WHERE avg_hour_of_day < 6 OR avg_hour_of_day > 22  -- Outside business hours
ORDER BY total_count DESC
LIMIT 30

In [16]:
%sql
-- User behavior summary

SELECT 
    principal_name,
    SUM(total_count) as total_events,
    COUNT(DISTINCT event_name) as event_types,
    AVG(avg_events_per_day) as avg_daily_activity,
    MAX(compartment_count) as compartments_accessed
FROM gitrepo.default.gold_user_behavior_baseline
GROUP BY principal_name
ORDER BY total_events DESC
LIMIT 20

In [17]:
%sql
-- Top anomalies (all types)

SELECT 
    run_time,
    anomaly_type,
    principal_name,
    event_name,
    compartment_name,
    ip_address,
    score,
    cnt_today,
    avg_60d,
    total_60d
FROM gitrepo.default.gold_audit_anomaly_incidents
ORDER BY score DESC
LIMIT 50

In [18]:
%sql
-- Top failed operations

SELECT 
    event_name,
    principal_name,
    compartment_name,
    ip_address,
    response_status,
    SUM(failure_count) as total_failures,
    MAX(last_failure) as most_recent_failure
FROM gitrepo.default.gold_failed_operations_summary
GROUP BY event_name, principal_name, compartment_name, ip_address, response_status
ORDER BY total_failures DESC
LIMIT 30

In [4]:
%sql
SELECT event_name, principal_name, response_status, COUNT(*) as count
FROM parquet.`/Volumes/gitrepo/default/git_oci_aidp_silver/audit_logs/data`
WHERE event_name = 'CreatePrivateIp'
  AND response_status != '200'
GROUP BY event_name, principal_name, response_status
ORDER BY count DESC
LIMIT 20